# Athermality, Example 2: a single-qubit Davies map

**Manuscript map.** Sec. II C, Fig. 3, Eqs. (19)-(24).

This notebook gives a self-contained reproduction of the quantum thermal
Mpemba example. It uses the two Bloch vectors and parameters quoted in
the Fig. 3 caption, evolves them analytically under a phase-covariant
Davies generator, and evaluates both quantum relative entropy and the
Petz-Renyi family.

**Learning goals**

1. Separate population relaxation from coherence decay.
2. Understand why a diagonal, highly excited state can thermalize faster
   than a less athermal coherent state.
3. Relate the late-time slopes to the complex Liouvillian spectrum.


## 1. Model and free-energy monotone

For $\hat H=(h/2)\sigma_z$, the fixed point is

$$
\hat\pi_\beta=\frac{e^{-\beta\hat H}}{\operatorname{Tr}e^{-\beta\hat H}}.
$$

The dimensionless excess free energy is exactly the relative entropy of
athermality,

$$
M[\hat\rho(t)] = S(\hat\rho(t)\Vert\hat\pi_\beta)
=\beta\,[F(\hat\rho(t))-F(\hat\pi_\beta)].
$$

The Davies map has one population rate $\Gamma_1$ and a conjugate pair of
coherence eigenvalues $-\Gamma_2\pm ih$. We choose
$\Gamma_1=2\Gamma_2$, the standard no-extra-dephasing relation.


In [ ]:
import sys
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as la

_HERE = Path.cwd().resolve()
for _candidate in [_HERE, *_HERE.parents]:
    if (_candidate / "tools" / "notebook_utils.py").exists():
        _TOOLS_DIR = _candidate / "tools"
        break
else:
    raise RuntimeError(
        "Could not locate tools/notebook_utils.py from cwd or its parents."
    )
if str(_TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(_TOOLS_DIR))

import notebook_utils as nu

warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=5, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7.2, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "font.size": 11,
})

BLUE = "#394F87"
ORANGE = "#E8792E"
GREEN = "#3A7D44"
PURPLE = "#7C4D9E"


In [ ]:
H_FREQUENCY = 10.0
BETA = 0.1
GAMMA_2 = 1.0
GAMMA_1 = 2.0 * GAMMA_2

r_random = np.array([0.221, 0.867, 0.206])
r_optimized = np.array([0.0, 0.0, 0.919])

def rho_from_bloch(vector):
    return (
        nu.I2
        + vector[0] * nu.X
        + vector[1] * nu.Y
        + vector[2] * nu.Z
    ) / 2

z_equilibrium = -np.tanh(BETA * H_FREQUENCY / 2)
pi = rho_from_bloch([0, 0, z_equilibrium])
rho_1 = rho_from_bloch(r_random)
rho_2 = rho_from_bloch(r_optimized)

for state in (pi, rho_1, rho_2):
    nu.validate_density(state)

print("Equilibrium Bloch z:", z_equilibrium)
print("Liouvillian spectrum:",
      [0, -GAMMA_1,
       -GAMMA_2 + 1j * H_FREQUENCY,
       -GAMMA_2 - 1j * H_FREQUENCY])


## 2. Analytic Davies evolution

In Bloch coordinates,

$$
r_z(t)=r_z^{\rm eq}+[r_z(0)-r_z^{\rm eq}]e^{-\Gamma_1 t},
\qquad
r_x(t)+ir_y(t)=[r_x(0)+ir_y(0)]e^{(-\Gamma_2+ih)t}.
$$

The optimized state has no overlap with the slow coherent pair. Its
athermality is initially larger because it is strongly population
inverted, but it decays with the faster population rate.


In [ ]:
times = np.linspace(0, 5, 2_001)

def evolve_bloch(vector, time):
    rotation = H_FREQUENCY * time
    transverse = np.exp(-GAMMA_2 * time)
    x = transverse * (
        vector[0] * np.cos(rotation)
        - vector[1] * np.sin(rotation)
    )
    y = transverse * (
        vector[0] * np.sin(rotation)
        + vector[1] * np.cos(rotation)
    )
    z = z_equilibrium + (
        vector[2] - z_equilibrium
    ) * np.exp(-GAMMA_1 * time)
    return rho_from_bloch([x, y, z])

states_1 = np.array([evolve_bloch(r_random, t) for t in times])
states_2 = np.array([evolve_bloch(r_optimized, t) for t in times])
monotone_1 = np.array([
    nu.quantum_relative_entropy(state, pi) for state in states_1
])
monotone_2 = np.array([
    nu.quantum_relative_entropy(state, pi) for state in states_2
])
tau = nu.crossing_time(times, monotone_2, monotone_1)

print(f"M(rho_1, 0) = {monotone_1[0]:.4f}")
print(f"M(rho_2, 0) = {monotone_2[0]:.4f}")
print(f"Crossing time = {tau:.4f}")
assert monotone_2[0] > monotone_1[0]
assert np.isfinite(tau)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))

axes[0].semilogy(times, monotone_1, color=PURPLE, lw=2,
                label=r"$\rho_1$: coherent")
axes[0].semilogy(times, monotone_2, color=PURPLE, lw=2, ls="--",
                label=r"$\rho_2$: optimized")
axes[0].axvline(tau, color="0.25", lw=1, ls=":",
                label=fr"$\tau={tau:.3f}$")
axes[0].set(xlabel=r"$t$", ylabel=r"$S(\rho(t)\Vert\pi_\beta)$",
            xlim=(0, 1.5), title="Quantum thermal Mpemba effect")
axes[0].legend()

spectrum = np.array([
    0,
    -GAMMA_1,
    -GAMMA_2 + 1j * H_FREQUENCY,
    -GAMMA_2 - 1j * H_FREQUENCY,
])
axes[1].scatter(spectrum.real[:2], spectrum.imag[:2],
                color=GREEN, label="populations")
axes[1].scatter(spectrum.real[2:], spectrum.imag[2:],
                color=ORANGE, label="coherences")
axes[1].axhline(0, color="0.6", lw=1)
axes[1].set(xlabel=r"$\operatorname{Re}\lambda$",
            ylabel=r"$\operatorname{Im}\lambda$",
            title="Davies spectrum")
axes[1].legend()
fig.tight_layout()
plt.show()


## 3. Petz-Renyi crossing times

The same pair of trajectories can be quantified with
$D_\alpha(\rho\Vert\pi_\beta)$. Nothing in the physical evolution
changes, but the inferred crossing time does.


In [ ]:
alphas = np.linspace(0.1, 2.0, 28)
sample = slice(None, None, 5)
sampled_times = times[sample]
crossing_times = []
for alpha in alphas:
    d1 = np.array([
        nu.petz_renyi(state, pi, alpha) for state in states_1[sample]
    ])
    d2 = np.array([
        nu.petz_renyi(state, pi, alpha) for state in states_2[sample]
    ])
    crossing_times.append(nu.crossing_time(sampled_times, d2, d1))

fig, ax = plt.subplots()
ax.plot(alphas, crossing_times, color=ORANGE, marker="o", ms=3)
ax.axvline(1, color="0.4", ls=":", lw=1)
ax.set(xlabel=r"Petz-Renyi parameter $\alpha$",
       ylabel=r"crossing time $\tau_\alpha$",
       title="Monotone-dependent crossing time")
plt.show()


## Takeaway

The diagonal state $\rho_2$ starts farther from the Gibbs state but has
exactly zero overlap with the slow coherence modes. It therefore relaxes
with $\Gamma_1=2\Gamma_2$ and crosses the coherent trajectory. The
complex eigenvalues are not decorative: their real part is the slow
envelope that the optimized state avoids.
